# Combining and Cleaning Fake Review Datasets

This notebook combines multiple fake review datasets into one clean dataset.

Final target column:
- `target = 1` means fake
- `target = 0` means real

Required packages:

```bash
pip install pandas numpy scikit-learn openpyxl
```


## 1. Imports and folders

In [ ]:
pip install pandas numpy scikit-learn openpyxl


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_DIRS = [
    Path("."),
    Path("data/raw")
]

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Current notebook folder:", Path(".").resolve())
print("Processed folder:", PROCESSED_DIR.resolve())

## 2. Dataset settings

Check that the file names below are exactly the same as the files in your folder.

The first dataset is set to:

`fake_reviews_dataset.csv`


In [ ]:
datasets = [
    {
        "name": "original_fake_reviews_dataset",
        "topic": "product_reviews",
        "file": "fake_reviews_dataset.csv",
        "text_col": "text_",
        "label_col": "label",
        "rating_col": "rating",
        "label_map": {
            "CG": "fake",
            "OR": "real"
        },
        "read_options": {}
    },
    {
        "name": "food_reviews",
        "topic": "food_reviews",
        "file": "Food_Reviews.csv",
        "text_col": "text",
        "label_col": "label",
        "rating_col": None,
        "label_map": {
            0: "real",
            1: "fake"
        },
        "read_options": {}
    },
    {
        "name": "labelled_yelp_dataset",
        "topic": "yelp_reviews",
        "file": "Labelled_Yelp_Dataset.csv",
        "text_col": "Review",
        "label_col": "Label",
        "rating_col": "Rating",
        "label_map": {
            -1: "fake",
            1: "real"
        },
        "read_options": {}
    },
    {
        "name": "restaurant_reviews",
        "topic": "restaurant_reviews",
        "file": "Restaurant_Reviews.csv",
        "text_col": "reviewContent",
        "label_col": "flagged",
        "rating_col": "rating",
        "label_map": {
            0: "real",
            1: "fake"
        },
        "read_options": {
            "sep": "\t"
        }
    },
    {
        "name": "restaurant_reviews_anonymized",
        "topic": "restaurant_reviews",
        "file": "restaurant_reviews_anonymized.csv",
        "text_col": "Review",
        "label_col": "Real",
        "rating_col": None,
        "label_map": {
            0: "fake",
            1: "real"
        },
        "read_options": {
            "encoding": "latin1"
        }
    },
    {
        "name": "reviews_dataset_excel",
        "topic": "restaurant_reviews",
        "file": "reviews_dataset.xlsx",
        "text_col": "text",
        "label_col": "label",
        "rating_col": None,
        "label_map": {
            0: "real",
            1: "fake"
        },
        "read_options": {}
    }
]

sentiment_only_datasets = [
    {
        "name": "hotel_reviews_sentiment_only",
        "topic": "hotel_reviews",
        "file": "Hotel_Reviews.csv",
        "positive_col": "Upside_Review",
        "negative_col": "Downside_Review",
        "rating_col": "Review_Score",
        "sentiment_col": "Sentiment",
        "read_options": {}
    }
]

## 3. Helper functions

In [ ]:
def find_file(filename):
    """
    Find a file in the current folder or in data/raw.
    This also checks a few small filename variations.
    """

    possible_names = [filename]
    file_path = Path(filename)

    if file_path.suffix == "":
        possible_names.append(filename + ".csv")
        possible_names.append(filename + ".xlsx")

    if file_path.suffix == ".csv":
        possible_names.append(file_path.stem)

    for folder in RAW_DIRS:
        for possible_name in possible_names:
            path = folder / possible_name
            if path.exists():
                return path

        # Case-insensitive fallback
        if folder.exists():
            for file in folder.iterdir():
                for possible_name in possible_names:
                    if file.name.lower() == possible_name.lower():
                        return file

    return None


def read_file(filename, read_options=None):
    """
    Read a CSV or Excel file.
    """

    read_options = read_options or {}
    path = find_file(filename)

    if path is None:
        raise FileNotFoundError(f"File not found: {filename}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, **read_options)

    options = {
        "on_bad_lines": "skip"
    }
    options.update(read_options)

    return pd.read_csv(path, **options)


def clean_text(value):
    """
    Clean review text for modelling.
    """

    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = re.sub(r"<.*?>", " ", value)
    value = re.sub(r"http\S+|www\S+", " ", value)
    value = re.sub(r"[^a-z0-9\s']", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def map_label(value, label_map):
    """
    Convert dataset-specific labels to fake or real.
    """

    if pd.isna(value):
        return np.nan

    if value in label_map:
        return label_map[value]

    value_as_string = str(value).strip()

    if value_as_string in label_map:
        return label_map[value_as_string]

    try:
        value_as_int = int(value)
        if value_as_int in label_map:
            return label_map[value_as_int]
    except:
        pass

    return np.nan


def standardize_labeled_dataset(settings):
    """
    Load one labeled dataset and convert it to the same structure.
    """

    df = read_file(settings["file"], settings.get("read_options"))

    text_col = settings["text_col"]
    label_col = settings["label_col"]
    rating_col = settings["rating_col"]
    label_map = settings["label_map"]

    missing_cols = [col for col in [text_col, label_col] if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{settings['file']} is missing columns: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    result = pd.DataFrame()
    result["text"] = df[text_col]
    result["clean_text"] = result["text"].apply(clean_text)
    result["label"] = df[label_col].apply(lambda value: map_label(value, label_map))
    result["target"] = result["label"].map({"real": 0, "fake": 1})

    if rating_col is not None and rating_col in df.columns:
        result["rating"] = df[rating_col]
    else:
        result["rating"] = np.nan

    result["source"] = settings["name"]
    result["topic"] = settings["topic"]

    return result


def clean_sentiment_only_hotel_dataset(settings):
    """
    Clean the hotel dataset separately because it does not have fake/real labels.
    """

    df = read_file(settings["file"], settings.get("read_options"))

    positive_col = settings["positive_col"]
    negative_col = settings["negative_col"]

    result = pd.DataFrame()

    result["text"] = (
        df[positive_col].fillna("").astype(str)
        + " "
        + df[negative_col].fillna("").astype(str)
    )

    result["clean_text"] = result["text"].apply(clean_text)
    result["sentiment"] = df[settings["sentiment_col"]]

    if settings["rating_col"] in df.columns:
        result["rating"] = df[settings["rating_col"]]
    else:
        result["rating"] = np.nan

    result["source"] = settings["name"]
    result["topic"] = settings["topic"]

    return result

## 4. Check whether all files can be found

In [ ]:
print("Checking files...\n")

for settings in datasets:
    found = find_file(settings["file"])

    if found is None:
        print("Missing:", settings["file"])
    else:
        print("Found:", settings["file"], "->", found)

print("\nChecking optional hotel sentiment file...\n")

for settings in sentiment_only_datasets:
    found = find_file(settings["file"])

    if found is None:
        print("Missing:", settings["file"])
    else:
        print("Found:", settings["file"], "->", found)

## 5. Load and standardize the labeled datasets

In [ ]:
labeled_frames = []

for settings in datasets:
    frame = standardize_labeled_dataset(settings)
    labeled_frames.append(frame)

    print(f"Loaded {settings['file']}: {frame.shape[0]} rows")

combined = pd.concat(labeled_frames, ignore_index=True)

print("\nCombined rows before cleaning:", combined.shape[0])
combined.head()

## 6. Clean the combined fake-review dataset

In [ ]:
combined = combined.dropna(subset=["label", "target"])
combined = combined[combined["clean_text"].str.len() > 0]

rows_before_duplicates = combined.shape[0]
combined = combined.drop_duplicates(subset=["clean_text"])
rows_after_duplicates = combined.shape[0]

combined["target"] = combined["target"].astype(int)

combined = combined[
    ["text", "clean_text", "label", "target", "rating", "source", "topic"]
]

print("Rows before duplicate removal:", rows_before_duplicates)
print("Rows after duplicate removal:", rows_after_duplicates)
print("Removed duplicates:", rows_before_duplicates - rows_after_duplicates)

combined.head()

## 7. Check the final dataset

In [ ]:
print("Final shape:", combined.shape)

print("\nLabel distribution:")
print(combined["label"].value_counts())

print("\nTarget distribution:")
print(combined["target"].value_counts())

print("\nRows per source:")
print(combined["source"].value_counts())

print("\nRows per topic:")
print(combined["topic"].value_counts())

## 8. Save the combined dataset

In [ ]:
combined_path = PROCESSED_DIR / "combined_fake_review_dataset_cleaned.csv"
combined.to_csv(combined_path, index=False)

print("Saved combined fake-review dataset to:")
print(combined_path.resolve())

## 9. Create train and test files

In [ ]:
train_df, test_df = train_test_split(
    combined,
    test_size=0.2,
    random_state=42,
    stratify=combined["target"]
)

train_path = PROCESSED_DIR / "fake_reviews_train.csv"
test_path = PROCESSED_DIR / "fake_reviews_test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Saved train file to:", train_path.resolve())
print("Saved test file to:", test_path.resolve())

## Notes for the report

Several datasets were combined because the first version of the project only used one dataset. The combined dataset now contains reviews from multiple domains, such as product reviews, Yelp reviews, restaurant reviews and food reviews.

All datasets were converted to the same structure. Review text was cleaned, duplicate reviews were removed and labels were standardized to fake and real.



# Label Check

Before using the cleaned dataset for modelling, the label mapping is checked.  
This is important because the original datasets use different label names, such as `CG`, `OR`, `0`, `1`, `-1` and `1`.

The goal of this check is to make sure that all labels are correctly converted to:

- `fake`
- `real`

And to:

- `target = 1` for fake reviews
- `target = 0` for real reviews

In [ ]:
# Check the basic structure of the final combined dataset

print("Dataset shape:", combined.shape)
print("\nColumns:")
print(combined.columns)

combined.head()

## 1. Check fake/real distribution

In [ ]:
# Check total fake/real distribution

print("Label distribution:")
print(combined["label"].value_counts())

print("\nTarget distribution:")
print(combined["target"].value_counts())

## 2. Check labels per dataset source

This table shows how many fake and real reviews each dataset contributes.

In [ ]:
# Check fake/real distribution per source

label_source_table = pd.crosstab(combined["source"], combined["label"])

label_source_table

## 3. Check labels per topic

This table shows the distribution of fake and real reviews per review topic.

In [ ]:
# Check fake/real distribution per topic

label_topic_table = pd.crosstab(combined["topic"], combined["label"])

label_topic_table

## 4. Show percentages per dataset

This makes it easier to see whether one dataset is very unbalanced.

In [ ]:
# Check percentages per source

label_source_percentage = pd.crosstab(
    combined["source"],
    combined["label"],
    normalize="index"
) * 100

label_source_percentage.round(2)

## 5. Manually inspect review examples

A few example reviews are shown for every dataset and label.  
This is used as a manual check to see whether the fake/real mapping seems logical.

In [ ]:
# Show example reviews for every source and label

for source in combined["source"].unique():
    print("\n" + "=" * 100)
    print("SOURCE:", source)
    print("=" * 100)

    for label in ["fake", "real"]:
        print(f"\nExamples labeled as {label}:")

        examples = combined[
            (combined["source"] == source) &
            (combined["label"] == label)
        ]["text"].head(3)

        if len(examples) == 0:
            print("No examples found.")
        else:
            for i, review in enumerate(examples, 1):
                print(f"\n{i}. {str(review)[:700]}")

## 6. Check original labels before mapping

The following code checks the original label values in the raw files.  
This helps verify whether the label mappings used in the cleaning step are correct.

In [ ]:
# Check original labels in the raw datasets

for settings in datasets:
    print("\n" + "=" * 100)
    print("DATASET:", settings["name"])
    print("FILE:", settings["file"])
    print("=" * 100)

    try:
        raw_df = read_file(settings["file"], settings.get("read_options"))
        label_col = settings["label_col"]

        print("Original label column:", label_col)
        print("\nOriginal label values:")
        print(raw_df[label_col].value_counts(dropna=False))

        print("\nLabel mapping used:")
        print(settings["label_map"])

    except Exception as e:
        print("Could not check this dataset.")
        print("Error:", e)

## 7. Create a label mapping overview

This table can be used in the report to explain how the original labels were converted.

In [ ]:
# Create an overview of the label mappings used

mapping_rows = []

for settings in datasets:
    for original_label, converted_label in settings["label_map"].items():
        mapping_rows.append({
            "dataset": settings["name"],
            "file": settings["file"],
            "original_label": original_label,
            "converted_label": converted_label,
            "target": 1 if converted_label == "fake" else 0
        })

label_mapping_overview = pd.DataFrame(mapping_rows)

label_mapping_overview

## 8. Save the label mapping overview

The label mapping overview is saved so it can be used as evidence in the report.

In [ ]:
# Save label mapping overview

mapping_path = PROCESSED_DIR / "label_mapping_overview.csv"
label_mapping_overview.to_csv(mapping_path, index=False)

print("Saved label mapping overview to:")
print(mapping_path.resolve())

## 9. Final check before modelling

This final check confirms that the cleaned dataset only contains valid fake/real labels.

In [ ]:
# Final validation checks

valid_labels = set(combined["label"].unique())
valid_targets = set(combined["target"].unique())

print("Unique labels:", valid_labels)
print("Unique targets:", valid_targets)

if valid_labels == {"fake", "real"} and valid_targets == {0, 1}:
    print("\nFinal check passed: labels and targets are valid.")
else:
    print("\nCheck needed: unexpected labels or targets found.")

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

combined.to_csv(PROCESSED_DIR / "combined_fake_review_dataset_cleaned.csv", index=False)

print("Saved combined dataset to:")
print((PROCESSED_DIR / "combined_fake_review_dataset_cleaned.csv").resolve())